# 0. Set up

Imagine that Princeton came to you for help recruiting its next incoming classes. Due to generous gifts from alumni, the school is expanding its sports programs to allow more students on each team and to add new sports like skiing. The university wants to identify current students who show evidence of sports-related interests, so it can look for high school students with similar interests to recruit.

The university knows that you're taking COS429, and they think you can help. They scraped thousands of images from the Instagram pages of students from the past 5 years, using only images marked “public,” and preprocessed this dataset for you to analyze. You have a sneaking suspicion that biases may creep into this analysis.

Your job in this assignment is to train a model that predicts whether an image contains a sports object, then measure its accuracy, precision, and recall overall and across demographic groups. You will analyze the model using **equal opportunity**: among students who truly show sports-related interest, the model should be similarly likely to identify them across groups. In metric terms, this means paying special attention to recall and false-negative-rate gaps.

More precisely, we're going to be using pre-trained features from a ResNet18 model trained on ImageNet to classify images in a dataset called [COCO (Common Objects in Context)](https://cocodataset.org/#home).

## Helper file

The starter bundle includes `bias_in_vision_models.py`, which mirrors selected metric and fairness-analysis TODOs from this notebook for staff-side checks. Your notebook analysis should train the sports baseline, compare accuracy, precision, and recall across gender and skin groups, then apply all three fairness interventions to the gender recall gap. The notebook and exported PDF are required for model outputs, tables, plots, and written analysis. Put all written responses in this notebook so they appear in your submitted PDF.


### Feature extraction

First, we need to extract features from the pretrained ResNet18 model. We have already computed this for you, but have also provided code for you to look through / recompute to better understand how this works. The necessary data can be found at **[annotations_trainval2014.zip](https://drive.google.com/file/d/11suBfKhkzOzshuM0MYMyKryezrDWU6s2/view?usp=sharing)**, and **[val2014.zip](https://drive.google.com/file/d/1hMOHWXluWHaqpgZ0zQgjtZYBvQJ6JG-l/view?usp=sharing)**.

The feature-extraction cells below are optional reference code. You do not need to run them for the assignment if you use the precomputed feature files linked in the next section.

In [3]:
!pip install pycocotools

!cp Downloads/annotations_trainval2014.zip .
!unzip annotations_trainval2014.zip

'cp' is not recognized as an internal or external command,
operable program or batch file.
'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
!cp Downloads/val2014.zip .
!unzip val2014.zip

'cp' is not recognized as an internal or external command,
operable program or batch file.
'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [1]:
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
import torchvision.models as models
from PIL import Image
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, precision_score, recall_score, f1_score

from pycocotools.coco import COCO
import json

In [6]:
annotation_file_path = 'annotations/instances_val2014.json'
coco = COCO(annotation_file_path)

loading annotations into memory...
Done (t=2.38s)
creating index...
index created!


In [7]:
# For feature extraction, we will ignore categories
# will stitch together categories in complete Dataset object
class COCOSubset(Dataset):

    def __init__(self, image_ids, root = "../../val2014", transform=None):
        self.images = image_ids
        self.root = root
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_id = self.images[idx]

        img_name = "{}/COCO_val2014_{:012d}.jpg".format(self.root, img_id)

        image = Image.open(img_name).convert("RGB")
        #label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return img_id, image


transform = transforms.Compose([transforms.Resize(size=(224, 224)),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                     std=[0.229, 0.224, 0.225])])


all_img_ids = coco.getImgIds()
dataset = COCOSubset(all_img_ids, root="val2014", transform=transform)
all_loader = DataLoader(dataset, batch_size=256, shuffle=False)

In [8]:
# Obtain ResNet for feature extraction
class Flatten(torch.nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return x.reshape(x.shape[0], -1)

model = models.resnet18(pretrained=True)
layers = list(model.children())
part_model = torch.nn.Sequential(*layers[:-1])

c:\Users\elean\miniconda3\envs\cos429\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\elean\miniconda3\envs\cos429\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\elean/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100.0%


In [10]:
# Do feature extraction and save intermediate pkls
all_ids = []
all_features = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

part_model.eval()
part_model.to(device)

with torch.no_grad():
  for t, (img_id, x) in enumerate(all_loader):
    x = x.to(device)

    feat = part_model(x).detach().cpu().numpy()
    all_features.append(feat)
    all_ids.append(img_id.numpy())

    if (t+1)%10==0:
      pickle.dump({"feat": np.concatenate(all_features), "idx":np.concatenate(all_ids)}, open("features{}.pkl".format((t+1)//10 ), 'wb+'))
      print(t, "features{}.pkl".format((t+1)//10))
      all_ids = []
      all_features = []


9 features1.pkl
19 features2.pkl
29 features3.pkl
39 features4.pkl
49 features5.pkl
59 features6.pkl
69 features7.pkl
79 features8.pkl


KeyboardInterrupt: 

In [ ]:
# Filter data subset by whether the image id has annotations
all_feat = []
all_idx= []

for i in range(1, 16):
  A = pickle.load(open("features{}.pkl".format(i ), 'rb'))
  all_feat.append(A["feat"])
  all_idx.append(A["idx"])


all_feat = np.concatenate(all_feat)
all_idx = np.concatenate(all_idx)


anno = pd.read_csv("images_val2014.csv", index_col=0)

valid_ids = set(anno.index)

mask = np.isin(all_idx, list(valid_ids))

coco_features = all_feat[mask]
coco_indices  = all_idx[mask]

np.save("coco_simplified_features.npy", coco_features)
np.save("coco_simplified_indices.npy", coco_indices)

In [ ]:
# Finally, make target and annotation arrays matched to simplified dataset
coco_ids = np.load("coco_simplified_indices.npy")

target_sport = []
cats_sport = []
genders = []
skins = []

for img_id in coco_ids:
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)

    cat_ids = [ann["category_id"] for ann in anns]
    cats = coco.loadCats(cat_ids)

    category_names = [c["name"] for c in cats]
    supercategory_names = [c["supercategory"] for c in cats]

    label_sport = 'sports' in supercategory_names
    cat_sport = category_names[supercategory_names.index('sports')] if label_sport else None

    gender = anno.loc[img_id, 'bb_gender']
    skin = anno.loc[img_id, 'bb_skin']

    target_sport.append(label_sport)
    cats_sport.append(cat_sport)
    genders.append(gender)
    skins.append(skin)


arrays = {
    "target_sport": np.array(target_sport),
    "cats_sport": np.array(cats_sport),
    "genders": np.array(genders),
    "skins": np.array(skins),
}

np.savez_compressed("coco_simplified_labels.npz", **arrays)

## How to Use Precomputed Features

We have already created files that contain data points with annotations. The notebook will require access to the following 3 files: **[coco_simplified_features.npy](https://drive.google.com/file/d/1BR_ybqK9pqW6JUpjA9uCFDp3eq6fAN0r/view?usp=sharing)**, **[coco_simplified_indices.npy](https://drive.google.com/file/d/1iW2ulLGvPC3RqJc6wo4Me5z-eOZIECic/view?usp=sharing)**, and **[coco_simplified_labels.npz](https://drive.google.com/file/d/1_nBSWrqDyfWOFcqSIindyZH34X0nR9rV/view?usp=sharing)**.

[Note: Dropping files into a Colab session repeatedly can be onerous, so one option to sidestep that is to put the files into your Google Drive, and mount your Drive in your Colab session. This option is available from the file explorer in Colab.]

After the required files are available in your Colab session, update the `TODO_filepath` strings in the next cell and run it to get data splits and loaders. Do not submit these data files to Gradescope. The loaders will not directly contain annotation values such as skin color, but the train-test splits themselves are matched with each other (so given an image id, you can obtain annotation values).

In [1]:
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
import torchvision.models as models
from PIL import Image
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, precision_score, recall_score, f1_score


In [2]:
# TODO (0): insert applicable filepaths
coco_features = np.load("coco_simplified_features.npy")
coco_ids = np.load("coco_simplified_indices.npy")
coco_labels = np.load("coco_simplified_labels.npz", allow_pickle=True)

target_sport = coco_labels["target_sport"]
cats_sport = coco_labels["cats_sport"]
genders = coco_labels["genders"]
skins = coco_labels["skins"]

(
    train_feat,
    val_feat,
    train_ids,
    val_ids,
    train_sport_target,
    val_sport_target,
    train_sport_cat,
    val_sport_cat,
    train_gender,
    val_gender,
    train_skin,
    val_skin,
) = train_test_split(
    coco_features,
    coco_ids,
    target_sport,
    cats_sport,
    genders,
    skins,
    test_size=0.4,
    random_state=42,
)

(
    val_feat,
    test_feat,
    val_ids,
    test_ids,
    val_sport_target,
    test_sport_target,
    val_sport_cat,
    test_sport_cat,
    val_gender,
    test_gender,
    val_skin,
    test_skin,
) = train_test_split(
    val_feat,
    val_ids,
    val_sport_target,
    val_sport_cat,
    val_gender,
    val_skin,
    test_size=0.5,
    random_state=42,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class COCOFeatures(Dataset):
    def __init__(self, img_ids, features, targets, device='cuda'):
        self.img_ids = img_ids
        self.features = features
        self.targets = targets
        self.device = device

    def __len__(self):
        return self.img_ids.shape[0]

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        feature = self.features[idx]
        target = self.targets[idx]

        return feature, target, img_id


In [4]:
batch_size = 64
num_workers = 2


def get_dataloader(ids, feat, target, batch_size=64, num_workers=2, shuffle=False):
    dataset = COCOFeatures(ids, feat, target)
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
    )
    return dataloader


train_sport_loader = get_dataloader(train_ids, train_feat, train_sport_target, shuffle=True)
val_sport_loader = get_dataloader(val_ids, val_feat, val_sport_target)
test_sport_loader = get_dataloader(test_ids, test_feat, test_sport_target)

# 1. Baseline model: sports objects

### Sports objects

#### Model Training.
Train a simple 2-3 layer neural network using the features from a pre-trained ResNet18 model to predict whether an image contains an object with the `sports` label. Everyone will use the sports model so that the fairness analysis and interventions are comparable across submissions.

In [5]:
from torch import nn

In [11]:
# TODO (1a): train a model

model_sports = nn.Sequential(
    nn.Flatten(start_dim=1), # (batch, 512, 1, 1) → (batch, 512)
    nn.Linear(train_feat.shape[1], 128),
    nn.ReLU(),
    nn.Linear(128, 1),
    nn.Sigmoid()
).to(device)

In [12]:
model_sports.eval()
with torch.no_grad():
  x = torch.Tensor(val_feat).to(device)
  val_scores = model_sports(x).squeeze().cpu().numpy()

#### Error analysis.

1. On the validation set, compute the overall accuracy of the sports model as well as the precision and recall scores.
2. Recompute these metrics over four subsets of the dataset: images labelled `female`, images labelled `male`, images labelled as having dark skin, and images labelled as having light skin. Some examples are labelled `Unsure`; include them in the overall metrics, but not in the female/male or dark/light subgroup rows.
3. Report the gender and skin gaps for accuracy, precision, and recall.
4. Compute the recall for different kinds of sports objects.

In [ ]:
# TODO (1a): overall accuracy on validation set, and precision and recall



# TODO (1a): accuracy, precision, recall on 'female', 'male' subsets
male_mask = np.where(val_gender == 'Male', 1, 0).astype(bool)
val_male_subset = val_scores[male_mask]
male_acc = None
male_prec = None
male_rec = None

print(male_acc, male_prec, male_rec, sep='\t')




# TODO (1a): accuracy, precision, recall on 'light skin', 'dark skin' subsets


In [ ]:
# TODO (1a): compute recall for different sports objects
# Hint: first find distinct categories within sports objects from train_sport_cat and val_sport_cat


# 2. Making the sports model more fair

## a. Equal-Opportunity Fairness Analysis

Use the principle of **equal opportunity** to analyze the sports model in about 150-200 words. For this assignment, equal opportunity means that among students who truly show evidence of sports-related interest, the model should be similarly likely to identify them across demographic groups.

Use your measured accuracy, precision, and recall. Focus especially on recall or false-negative-rate gaps across gender, since a false negative means a potentially interested student is missed. You should also discuss the skin-group metrics you measured.

**Sports Model**
TODO (2a)

## b. Intervening on the sports model

In this section, apply **all three** interventions below to the sports model: reweighting, changing the loss function, and changing the prediction threshold.

Use **gender recall gap** as the shared target metric. This is equivalent to targeting the gender false-negative-rate gap, because lower recall means more false negatives among students who truly show sports-related interest. Continue to report skin-group metrics as secondary diagnostics.

For every intervention, include your code, a before/after metrics table, and a short written interpretation in this notebook so it appears in your submitted PDF. Repeat the same before/after error analysis:

1. Overall accuracy, precision, and recall.
2. Female and male accuracy, precision, and recall.
3. Dark-skin and light-skin accuracy, precision, and recall.
4. Gender and skin gaps for accuracy, precision, and recall.
5. Sports-object recalls when useful for explaining the result.

Success means reducing the gender recall gap. Other metrics may improve, stay similar, or get worse. After each intervention, write a short reflection explaining what worked and what did not. If the target gap decreased, discuss the tradeoffs you observed. If it did not, report that clearly and explain what changed instead.

When training models to be more fair, we can intervene in different stages of the machine learning pipeline:

1. Pre-processing algorithms intervene by adjusting the input data. One method is to reweight different data points.
2. In-processing algorithms intervene during training, for example by adding a fairness term to the loss function.
3. Post-processing algorithms use a trained model and adjust the prediction boundary in different ways.

#### Reweighting a dataset.

A simple way to reweight different elements during training is to use the `weight` parameter within the `BCELoss()` (https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html).

For this assignment, use the same train/validation split as the baseline. Identify the gender group with lower baseline recall in the sports model, then upweight positive sports training examples from that group. For example, if the sports model has lower recall on images labelled female, give positive female sports examples a weight $\alpha > 1$ and keep other examples at weight 1.

Repeat the before/after error analysis described above, and report whether the gender recall gap decreased.

In [ ]:
# TODO (2b option 1): modify model training to use weight parameter in BCELoss
# Hint: the dataloader returns img_id, which you can use to match samples to demographics


Given what you just saw about the effects of reweighting, write about 50 words explaining whether this made the sports model more fair under equal opportunity.

TODO (2b reweighting)

#### Changing the loss function

We can add a fairness term to a standard loss function during training, to incentivize learning of a model that prioritizes accuracy and fairness. To have the loss be differentiable, use the probabilistic outputs of the classifier.

For this assignment, add a penalty for the absolute difference between the female and male soft recalls. If the target labels are $y_i \in \{0, 1\}$, the sensitive attribute is $a_i \in \{\text{female, male}\}$, and the model outputs scores $s_i$, the probability of predicting the positive class is $p_i = \sigma(s_i)$. A soft recall-difference penalty is:

$$
L_{\text{fair}} = \left|\frac{\sum_{i | (a_i = \text{female and }y_i = 1)}p_i}{\sum_{i | (a_i = \text{female and }y_i = 1)}1 + \epsilon} - \frac{\sum_{i | (a_i = \text{male and }y_i = 1)}p_i}{\sum_{i | (a_i = \text{male and }y_i = 1)}1 + \epsilon}\right|
$$

Thus, the overall loss becomes:

$$
L_\text{overall} = BCELoss() + \alpha L_{\text{fair}}
$$

where $\alpha$ is a hyperparameter that trades off between prediction performance and fairness. Try at least one reasonable value of $\alpha$, and report the value you used. Repeat the before/after error analysis described above, and report whether the gender recall gap decreased.

In [ ]:
# TODO (2b option 2): implement this loss function; will also require matching id to demographics


Given what you just saw about the effects of changing the loss function, write about 50 words explaining whether this made the sports model more fair under equal opportunity.

TODO (2b loss function)

#### Changing the prediction threshold

When considering different performances on different demographic groups, one method to improve fairness is to pick different thresholds for different groups. For example, you could choose a lower threshold for the gender group with lower baseline recall.

For this assignment, choose female and male thresholds that target the gender recall gap. Report what happens to precision and accuracy. If you use group-specific thresholds, say explicitly that the post-processing rule uses the sensitive attribute at decision time.

Repeat the before/after error analysis described above, and report whether the gender recall gap decreased.

In [ ]:
# TODO (2b option 3): redo error analysis with group-by-group thresholds

Given what you just saw about the effects of changing the prediction threshold, write about 50 words explaining whether this made the sports model more fair under equal opportunity.

TODO (2b thresholding)